In [4]:
from transformers import AutoProcessor, AutoModelForMultimodalLM
import torch, accelerate, torchvision
import os

from dotenv import load_dotenv
load_dotenv("../../api_key.env")
hf_token = os.getenv("HF_TOKEN")

In [24]:
MODEL_ID = "google/gemma-4-E2B-it"

os.environ["PYTORCH_MPS_HIGH_WATERMARK_RATIO"] = "0.0"

# 1. Load the processor and the model
# AutoModelForMultimodalLM handles image, audio, and text inputs
# device = "cuda" if torch.cuda.is_available() else "cpu"

# device = "mps" if torch.backends.mps.is_available() else "cpu"

device = torch.device("mps")
print(device)

model = AutoModelForMultimodalLM.from_pretrained(
    MODEL_ID, 
    dtype=torch.float16,
    device_map={"*:": "mps"}
)

processor = AutoProcessor.from_pretrained(MODEL_ID)

mps


error: nothing to repeat at position 1

In [20]:
# 2. Structure the prompt with a multimodal input (e.g., an ad film frame)
messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "url": "https://lh3.googleusercontent.com/pw/AP1GczO3okabc1lPmVdkvrdunvXf0G16h4GhOqL535wJyLKYyPCy7YWHul0TXhkCIincP5tJAkhX8osJCIB6BTY1jR8S6vpHD_hjNqXTiSNA3rqjdq3dwprzTPX95pfalIc_YKo4e3P82jaZrVv3ikxy8D4GlQ=w1152-h864-s-no-gm?authuser=0"},
            {"type": "text", "text": "Analyze this ad frame. Does it have a clear visual hook?"}
        ]
    }
]

# 3. Process the inputs and format the chat template
inputs = processor.apply_chat_template(
    messages, 
    tokenize=True, 
    return_dict=True, 
    return_tensors="pt", 
    add_generation_prompt=True,
    enable_thinking=True # Activates Gemma 4's internal reasoning process
)

inputs = {k: v.to(device) if isinstance(v, torch.Tensor) else v for k, v in inputs.items()}

In [21]:
# 4. Generate the output
output = model.generate(
            **inputs, 
            max_new_tokens=512
            )

/Users/apple/Code/utmtvenv/lib/python3.11/site-packages/transformers/generation/utils.py:2528: UserWarning: You are calling .generate() with the `input_ids` being on a device type different than your model's device. `input_ids` is on mps, whereas the model is on meta. You may experience unexpected behaviors or slower generation. Please make sure that you have put `input_ids` to the correct device by calling for example input_ids = input_ids.to('meta') before running `.generate()`.
  warnings.warn(


RuntimeError: Tensor on device meta is not on the expected device mps:0!

In [ ]:
# 5. Decode the response
input_len = inputs["input_ids"].shape[-1]
response = processor.decode(output[0][input_len:], skip_special_tokens=True)

print(response)